In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "8"

In [2]:
import openai
from tqdm.auto import tqdm
import time

/mount/arbeitsdaten/asr-2/vaethdk/virtualenvs/cts_en/lib64/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
openai.api_key = os.environ["OPENAI_API_KEY"]

In [5]:
import sys
sys.path.append('../../..')
print(os.path.realpath("."))

from data.dataset import ReimburseGraphDataset, StandardGraphDataset, DataAugmentationLevel, NodeType, DialogNode, Question

/mount/arbeitsdaten41/projekte/asr-2/vaethdk/cts_newcodebase_rollback/conversational-tree-search/generation/onboarding/chatgpt


In [ ]:
onboard_human_data = StandardGraphDataset('en/onboarding/train_graph.json', 'en/onboarding/train_answers.json', True, DataAugmentationLevel.NONE, augmentation_path=None, resource_dir='../../../resources')
# reimburse_human_data = ReimburseGraphDataset('en/reimburse/train_graph.json', 'en/reimburse/train_answers.json', True, DataAugmentationLevel.NONE, augmentation_path=None, resource_dir='../../../resources')

===== Dataset Statistics =====
- files:  en/onboarding/train_graph.json en/onboarding/train_answers.json
- synonyms: True
- depth: 12  - degree: 9
- answers: 43
- questions: 141
- loaded original data: True
- loaded generated data: False
- question limit: 0  - maximum loaded:  4
- answer limit: 0  - maximum loaded:  1


In [ ]:
def prompt_v2(node_text: str, num_questions: int):
    return f"""Generate {num_questions} questions about the given facts: "{node_text}"""

def api_prompt_v2(prompt: str):
    return [
        {"role": "system", "content": "You are a truthful assistant, generating diverse FAQ-style questions given some facts. The generated questions should be answerable using the given fact only, without additional knowledge. The questions should also be short and human-like. Try to vary the amount of information between questions. Present the results in a numbered list."},
        {"role": "user", "content": prompt},
    ]

def api_completion_v2(node_text: str, num_questions: int):
    return openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=api_prompt_v2(prompt_v2(node_text, num_questions))
    )

In [14]:
# TODO integrate NER extraction

def prompt_v3(node_text: str, num_questions: int):
    return f"""Generate {num_questions} questions about the given facts: "{node_text}"""

def prompt_v3_ner(answer_text: str, ner: str, num_questions: int):
    return f"""Generate {num_questions} questions about the entity "{ner}" from the fact: "{answer_text}" """

def api_prompt_v3(prompt: str):
    return [
        {"role": "system", "content": "You are a truthful assistant, generating diverse FAQ-style questions given some facts. The generated questions should be answerable using the given fact only, without additional knowledge. The questions should also be short and human-like. Try to vary the amount of information between questions. Present the results in a numbered list."},
        {"role": "user", "content": prompt},
    ]

def api_completion_v3(node_text: str, num_questions: int):
    return openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=api_prompt_v3(prompt_v3(node_text, num_questions))
    )

def api_completion_v3_ner(answer_text: str, ner: str, num_questions: int):
    return openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=api_prompt_v3(prompt_v3_ner(answer_text, ner, num_questions))
    )


In [6]:
import stanza
nlp = stanza.Pipeline('en', processors='tokenize,ner', device="cuda:0")

2023-10-12 20:27:59 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2023-10-12 20:28:00 INFO: Loading these models for language: en (English):
| Processor | Package   |
-------------------------
| tokenize  | combined  |
| ner       | ontonotes |

2023-10-12 20:28:00 INFO: Using device: cuda:0
2023-10-12 20:28:00 INFO: Loading: tokenize
2023-10-12 20:28:02 INFO: Loading: ner
2023-10-12 20:28:03 INFO: Done loading processors!


In [7]:
from statistics import mean

nodes_with_ner = 0
nodes_without_ner = 0
avg_node_ner = []

for node in tqdm(onboard_human_data.nodes_by_type[NodeType.INFO]):
    context = nlp(node.text)
    if len(context.ents) > 0:
        nodes_with_ner += 1
        avg_node_ner.append(len(context.ents))
    else:
        nodes_without_ner += 1

print("TOTAL INFO NODES", len(onboard_human_data.nodes_by_type[NodeType.INFO]))
print("NODES WITH NER", nodes_with_ner)
print("NODES WITHOUT NER", nodes_without_ner)
print("AVG NER PER NODE WITH NER", mean(avg_node_ner))

100%|██████████| 59/59 [00:07<00:00,  7.85it/s]

TOTAL INFO NODES 59
NODES WITH NER 46
NODES WITHOUT NER 13
AVG NER PER NODE WITH NER 2.5869565217391304


In [9]:
avg_node_sentence_length = []

for node in onboard_human_data.nodes_by_type[NodeType.INFO]:
    avg_node_sentence_length.append(node.text.count("."))

print("MAX #SENTENCES PER NODE", max(avg_node_sentence_length))
print("AVG #SENTENCES PER NODE", mean(avg_node_sentence_length))

MAX #SENTENCES PER NODE 4
AVG #SENTENCES PER NODE 1.5084745762711864


In [10]:
from typing import List, Tuple

def extract_ner_sentences(node: DialogNode) -> List[Tuple[str, str]]:
    """
    Extract all sentences from node text that mention NER's.
    Returns them as a list of tuples, where each tuple contains
        1. the name of the entity
        2. the sentence containing that entity
    """
    results = []
    context = nlp(node.text)
    entities = context.ents
    for entity in entities:
        start_idx = entity.start_char
        end_idx = entity.end_char
        # expand start index to beginning of sentence
        while start_idx > 0 and node.text[start_idx-1] != ".":
            start_idx -= 1
        # expand end index to end of sentence
        while end_idx < len(node.text) and node.text[end_idx-1] != ".":
            end_idx += 1
        results.append((entity.text, node.text[start_idx:end_idx]))
    return results

In [12]:
# find a testing candidate
for node in onboard_human_data.nodes_by_type[NodeType.INFO]:
    results = extract_ner_sentences(node)
    if len(results) > 1:
        print(results)
        break

[('one', "For short stays, consider looking into one of the city's many hostels or hotels, which can be booked on a per night basis."), ('40-150 euros', ' Prices should start in the rnage of 40-150 euros per night.')]


In [15]:
def parse_output(result):
    result_strings = result.get('choices')[0].get("message").get("content").split('\n')
    questions = []
    unnumbered_questions = []
    question_idx = 1
    for question in result_strings:
        question = question.strip()
        if question.startswith(f"{question_idx}."):
            questions.append(question.strip(f"{question_idx}.").strip())
        else:
            unnumbered_questions.append(question)
        question_idx += 1
    return questions, unnumbered_questions

In [31]:
import time
import traceback

system = """You are a helpful assistant creating a list of diverse FAQ-style questions from given facts.
Only generate questions that can be answered by the given facts, without any external knowledge.
Use casual language.
Prefer short questions.
Order the generated paraphrases in a numbered list."""

def user(answer_text: str, num_paraphrases: int) -> str:
    return f'Generate {num_paraphrases} short and diverse FAQ-style questions from the fact: "{answer_text}"'

def user_ner(answer_text: str, ner: str, num_paraphrases: int) -> str:
    return f'Generate {num_paraphrases} short and diverse FAQ-style questions about the entity "{ner}" from the fact: "{answer_text}"'


NUM_QUESTIONS = 10
NUM_QUESTIONS_PER_SENTENCE = 3
TEMPERATURE = 0.7
MAX_NEW_TOKENS = 1024

generated_data = {}
generated_data_unnumbered = {}

for node in tqdm(onboard_human_data.nodes_by_type[NodeType.INFO]):
    # use dict indexed by generated text to filter out duplicates
    all_generations = {}
    
    # extract NERs
    named_entities = extract_ner_sentences(node)
    # print(named_entities)

    # Generate questions with NER sentences only, make asking about NER a requirement
    for entity, sentence in named_entities:
        # print("- ENTITY", entity)
        done = False
        while not done:
            try:
                # prompt = prompt_v3_ner(node.text, entity, NUM_QUESTIONS_PER_SENTENCE)
                gen = api_completion_v3_ner(node.text, entity, NUM_QUESTIONS_PER_SENTENCE)
                questions, unnumbered_questions = parse_output(gen)

                for question in questions:
                    key = str(time.time()).replace(".", "")
                    all_generations[key] = {
                        "key": key,
                        "context": "ner",
                        "entity": entity,
                        "dialog_node_key": node.key,
                        "node_text": node.text,
                        "text": question
                    }
                for question in unnumbered_questions:
                    key = str(time.time()).replace(".", "")
                    generated_data_unnumbered[key] = {
                        "key": key,
                        "context": "ner",
                        "entity": entity,
                        "dialog_node_key": node.key,
                        "node_text": node.text,
                        "text": question
                    }
                done = True
            except:
                traceback.print_exc()
                done = True
                print("waiting...")
                time.sleep(15)
    # Generate questions with whole context
    num_node_level_questions = max(NUM_QUESTIONS_PER_SENTENCE, NUM_QUESTIONS - len(named_entities) * NUM_QUESTIONS_PER_SENTENCE)
    # print("NUM GENERIC", num_node_level_questions)
    done = False
    while not done:
        try:
            gen = api_completion_v3(node.text, num_node_level_questions)
            questions, unnumbered_questions = parse_output(gen)

            for question in questions:
                    key = str(time.time()).replace(".", "")
                    all_generations[key] = {
                        "key": key,
                        "context": "node",
                        "dialog_node_key": node.key,
                        "node_text": node.text,
                        "text": question
                    }
            for question in unnumbered_questions:
                key = str(time.time()).replace(".", "")
                generated_data_unnumbered[key] = {
                    "key": key,
                    "context": "node",
                    "dialog_node_key": node.key,
                    "node_text": node.text,
                    "text": question
                }
            done = True
        except:
            traceback.print_exc()
            done = True
            print("waiting...")
            time.sleep(15)
    # filter out duplicates
    uniques = set()
    for question_key in all_generations:
        question = all_generations[question_key]
        if question['text'].lower() in uniques:
            continue # skip duplicate
        else:
            uniques.add(question['text'].lower())
            generated_data[question['key']] = question


  0%|          | 0/59 [00:00<?, ?it/s]

[('one', "For short stays, consider looking into one of the city's many hostels or hotels, which can be booked on a per night basis."), ('40-150 euros', ' Prices should start in the rnage of 40-150 euros per night.')]
- ENTITY one
- ENTITY 40-150 euros
NUM GENERIC 3


  0%|          | 0/59 [00:19<?, ?it/s]


In [29]:
print(uniques)

{'where can i find accommodations for short stays in the city?', 'where can i find affordable accommodations for short stays in the city?', "what is the price range for per night bookings in the city's hostels or hotels?", 'how much do hostels or hotels typically charge per night in the city?', 'what types of accommodations are available for short stays in the city?', 'how much does it cost per night to book a hostel or hotel in the city?', 'what is the price range for booking a hostel or hotel in the city on a per night basis?', "what is the minimum and maximum price per night for accommodations in the city's hostels or hotels?", "what is the price range for a short stay in one of the city's hostels or hotels?"}


In [32]:
import json
with open("../../../resources/en/onboarding/generated/chatgpt/train_questions_v3.json", "w") as f:
    json.dump(generated_data, f)

with open("../../../resources/en/onboarding/generated/chatgpt/train_questions_v3_unnumbered.json", "w") as f:
    json.dump(generated_data_unnumbered, f)

# Generate Answers

In [8]:
onboard_human_data = StandardGraphDataset('en/onboarding/train_graph.json', 'en/onboarding/train_answers.json', True, DataAugmentationLevel.NONE, augmentation_path=None, resource_dir='../../../resources')
reimburse_human_data = ReimburseGraphDataset('en/reimburse/train_graph.json', 'en/reimburse/train_answers.json', True, DataAugmentationLevel.NONE, augmentation_path=None, resource_dir='../../../resources')

===== Dataset Statistics =====
- files:  en/onboarding/train_graph.json en/onboarding/train_answers.json
- synonyms: True
- depth: 12  - degree: 9
- answers: 43
- questions: 141
- loaded original data: True
- loaded generated data: False
- question limit: 0  - maximum loaded:  4
- answer limit: 0  - maximum loaded:  1
===== Dataset Statistics =====
- files:  en/reimburse/train_graph.json en/reimburse/train_answers.json
- synonyms: True
- depth: 20  - degree: 13
- answers: 312
- questions: 279
- loaded original data: True
- loaded generated data: False
- question limit: 0  - maximum loaded:  7
- answer limit: 0  - maximum loaded:  9


In [38]:
def prompt(node_text: str, answer_text: str, num_paraphrases: int):
    return f"""Generate {num_paraphrases} answer paraphrases for the answer "{answer_text}" to the question: "{node_text}"""

def api_prompt(prompt: str):
    return [
        {"role": "system", "content": "You are a truthful assistant, generating diverse paraphrases for a prototypical answer to a given question. The generated answer paraphrases should be human-like and preferably short. Present the results in a numbered list."},
        {"role": "user", "content": prompt},
    ]

def api_completion(node_text: str, answer_text: str, num_paraphrases: int):
    return openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=api_prompt(prompt(node_text, answer_text, num_paraphrases))
    )

In [39]:
from collections import defaultdict
import traceback

NUM_PARAPHRASES = 10 

generated = defaultdict(lambda: set())
generated_unnumbered = defaultdict(lambda: set())

num_generated = 0
num_generated_unnumbered = 0

for idx, node in tqdm(enumerate(onboard_human_data.nodes_by_type[NodeType.QUESTION])):
    for answer in node.answers:
        done = False
        while not done:
            try:
                print(api_prompt(prompt(node.text, answer.text, NUM_PARAPHRASES)))
                response = api_completion(node.text, answer.text, NUM_PARAPHRASES)
                answers, unnumbered_answers = parse_output(response)

                generated[answer.text.strip().lower()] = generated[answer.text.strip().lower()].union(answers)
                generated_unnumbered[answer.text.strip().lower()] = generated_unnumbered[answer.text.strip().lower()].union(unnumbered_answers)

                num_generated += len(answers)
                num_generated_unnumbered += len(unnumbered_answers)

                if idx % 10 == 0:
                    print(f"Generated: {num_generated}, Unnumbered: {num_generated_unnumbered}")
                
                done = True
            except:
                traceback.print_exc()
                done = True
                print("waiting...")
                time.sleep(15)

0it [00:00, ?it/s]

[{'role': 'system', 'content': 'You are a truthful assistant, generating diverse paraphrases for a prototypical answer to a given question. The generated answer paraphrases should be human-like and preferably short. Present the results in a numbered list.'}, {'role': 'user', 'content': 'Generate 10 answer paraphrases for the answer "Finding Housing" to the question: "Hi I\'m the Stuttgart Help-Bot! My goal is to help your move to Germany and especially to Stuttgart go as smoothly as possible!\n What would you like to know about?'}]


0it [00:13, ?it/s]

Generated: 10, Unnumbered: 0


In [40]:
generated

defaultdict(<function __main__.<lambda>()>,
            {'finding housing': {'"Discovering Housing Opportunities"',
              '"Exploring Housing Options"',
              '"Finding a Place to Live"',
              '"Hunting for a Place in Stuttgart"',
              '"Inquiries About Housing in Stuttgart"',
              '"Looking for a Place to Stay"',
              '"Scouting for a Residence"',
              '"Searching for Accommodation"',
              '"Seeking Housing Solutions"',
              '"Seeking a Home in Stuttgart"'}})

In [28]:
generated_unnumbered

defaultdict(<function __main__.<lambda>()>, {})

In [24]:
api_prompt(prompt(node_text, answer_text, num_paraphrases))

NameError: name 'node_text' is not defined

# Better Answer Generation

In [9]:
onboard_human_data = StandardGraphDataset('en/onboarding/train_graph.json', 'en/onboarding/train_answers.json', True, DataAugmentationLevel.NONE, augmentation_path=None, resource_dir='../../../resources')
reimburse_human_data = ReimburseGraphDataset('en/reimburse/train_graph.json', 'en/reimburse/train_answers.json', True, DataAugmentationLevel.NONE, augmentation_path=None, resource_dir='../../../resources')

===== Dataset Statistics =====
- files:  en/onboarding/train_graph.json en/onboarding/train_answers.json
- synonyms: True
- depth: 12  - degree: 9
- answers: 43
- questions: 141
- loaded original data: True
- loaded generated data: False
- question limit: 0  - maximum loaded:  4
- answer limit: 0  - maximum loaded:  1
===== Dataset Statistics =====
- files:  en/reimburse/train_graph.json en/reimburse/train_answers.json
- synonyms: True
- depth: 20  - degree: 13
- answers: 312
- questions: 279
- loaded original data: True
- loaded generated data: False
- question limit: 0  - maximum loaded:  7
- answer limit: 0  - maximum loaded:  9


In [6]:
def prompt(node_text: str, answer_text: str, num_paraphrases: int):
    return f"""Generate {num_paraphrases} paraphrases for the response "{answer_text}" to the question {node_text}"""

def api_prompt(prompt: str):
    return [
        {"role": "system", "content": "You are generating semantically similar paraphrases for a given response to some question. The generated response paraphrases should be human-like and short, using frequently used words and phrases only. Present the results in a numbered list."},
        {"role": "user", "content": prompt},
    ]

def api_completion(node_text: str, answer_text: str, num_paraphrases: int):
    return openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=api_prompt(prompt(node_text, answer_text, num_paraphrases))
    )

In [19]:
def prompt(node_text: str, answer_text: str, num_paraphrases: int):
    return f"""Generate {num_paraphrases} options for shortening the response "{answer_text}" to the question {node_text}"""

def api_prompt_keywords(prompt: str):
    return [
        {"role": "system", "content": "You are shortening a given response to some question into a keyword-like prompt. Present the results in a numbered list."},
        {"role": "user", "content": prompt},
    ]

def api_completion_keywords(node_text: str, answer_text: str, num_paraphrases: int):
    return openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=api_prompt_keywords(prompt(node_text, answer_text, num_paraphrases))
    )

In [20]:
def parse_output(result):
    result_strings = result.get('choices')[0].get("message").get("content").split('\n')
    questions = []
    unnumbered_questions = []
    question_idx = 1
    for question in result_strings:
        question = question.strip()
        if question.startswith(f"{question_idx}."):
            questions.append(question.strip(f"{question_idx}.").strip())
        else:
            unnumbered_questions.append(question)
        question_idx += 1
    return questions, unnumbered_questions

In [21]:
from collections import defaultdict
import traceback

NUM_PARAPHRASES = 5
NUM_KEYWORD_PARAPHRASES = 5

generated = defaultdict(lambda: set())
generated_unnumbered = defaultdict(lambda: set())

num_generated = 0
num_generated_unnumbered = 0

for idx, node in tqdm(enumerate(onboard_human_data.nodes_by_type[NodeType.QUESTION])):
    for answer in node.answers:
        done = False
        while not done:
            try:
                print(api_prompt_keywords(prompt(node.text, answer.text, NUM_KEYWORD_PARAPHRASES)))
                response = api_completion_keywords(node.text, answer.text, NUM_KEYWORD_PARAPHRASES)
                answers, unnumbered_answers = parse_output(response)

                generated[answer.text.strip().lower()] = generated[answer.text.strip().lower()].union(answers)
                generated_unnumbered[answer.text.strip().lower()] = generated_unnumbered[answer.text.strip().lower()].union(unnumbered_answers)

                num_generated += len(answers)
                num_generated_unnumbered += len(unnumbered_answers)

                if idx % 10 == 0:
                    print(f"Generated: {num_generated}, Unnumbered: {num_generated_unnumbered}")
                
                done = True
            except:
                traceback.print_exc()
                done = True
                print("waiting...")
                time.sleep(15)
        break
    break

import pprint
pprint.pprint(generated)

0it [00:00, ?it/s]

[{'role': 'system', 'content': 'You are shortening a given response to some question into a keyword-like prompt. Present the results in a numbered list.'}, {'role': 'user', 'content': 'Generate 5 options for shortening the response "Finding Housing" to the question Hi I\'m the Stuttgart Help-Bot! My goal is to help your move to Germany and especially to Stuttgart go as smoothly as possible!\n What would you like to know about?'}]


0it [00:04, ?it/s]

Generated: 5, Unnumbered: 0
defaultdict(<function <lambda> at 0x7f9ce439ed40>,
            {'finding housing': {'Accommodation in Germany',
                                 'Finding a Home',
                                 'Housing Search',
                                 'Housing in Stuttgart',
                                 'Stuttgart Housing'}})
